# 97 · Avaliação de um modelo já desenvolvido (do MLflow)

Este notebook **não treina nada**. Ele pega um modelo de PD já desenvolvido e salvo no MLflow, aplica na sua base, usa as **faixas de rating que você informar** e avalia o modelo com as mesmas tabelas e gráficos do pipeline desafiante (semáforo, recalibração, volumetria, estabilidade).

Dois tipos de modelo, escolhidos em `TIPO_MODELO`:

| tipo | quando usar | como escora |
|---|---|---|
| `"logistica"` | regressão logística (por exemplo, sobre variáveis em WoE), inclusive com coeficientes digitados | fórmula da PD calculada no próprio Spark, a partir do intercepto e dos coeficientes |
| `"generico"` | qualquer outro modelo do MLflow: árvores, boosting (LightGBM, XGBoost), pipelines do scikit-learn | o próprio modelo, chamado em lotes pandas nos workers (`mapInPandas`), sempre pela **probabilidade** (nunca pela classe 0/1) |

| passo | o que faz | de onde vem |
|---|---|---|
| 1. base | lê o parquet ou a tabela, leva as datas ao dia 01, separa dev e OOT e tira as safras imaturas | novo + `01_entrada` |
| 2. modelo | logística: extrai intercepto e coeficientes; genérico: carrega o modelo pela biblioteca original | novo |
| 3. escoragem | PD de cada contrato e conferência (logística: contra o próprio modelo; genérico: sanidade da PD) | novo |
| 3b. calibração no dev | ajusta o nível da PD no dev (intercepto, padrão; ou Platt) e aplica a mesma regra em todas as safras | novo |
| 4. faixas | aplica as faixas que você informou; o valor de cada rating é a taxa de maus observada no dev | novo + `05_regua` |
| 5. métricas por período | KS, AUC, Gini, taxa de maus, PD média e PSI do score, no dev e no OOT | novo |
| 6. monitoramento | semáforo por rating e safra, recalibração (só no OOT), volumetria, painel de estabilidade, ponte | `05_regua` (`monitora_regua`) |

**Exemplo da escoragem.** Intercepto −3,0; duas variáveis WoE com coeficientes 0,8 e 0,5; um contrato com WoE 0,4 e −0,2:

```
margem = −3,0 + 0,8 × 0,4 + 0,5 × (−0,2) = −3,0 + 0,32 − 0,10 = −2,78
PD     = 1 / (1 + e^2,78) = 1 / (1 + 16,1) = 0,058  →  5,8%
```

**Base de entrada:** só os períodos dev e OOT. Colunas: safra (mês de referência), alvo (0/1), período (dev/oot) e as variáveis com os nomes e o formato que o modelo espera (por exemplo, já em WoE, se o modelo foi treinado em WoE). As demais colunas são ignoradas.

**Cluster:** roda em cluster clássico ou compartilhado e no serverless. No tipo genérico, a biblioteca do modelo (lightgbm, xgboost...) precisa estar instalada numa versão compatível com a do treino; o MLflow registra essa versão junto com o modelo (`requirements.txt` do artefato).

## 1. Carrega as funções do pipeline

In [ ]:
%run ./00_config

In [ ]:
%run ./01_entrada

In [ ]:
%run ./05_regua

## 2. Parâmetros

**Calibração no dev.** A PD do modelo pode ter sido treinada em outro período, com outro nível de inadimplência. Para comparar o nível da PD do modelo com a régua, a PD é calibrada no dev e a mesma regra é aplicada ao OOT (`CALIBRACAO_DEV`):

| valor | o que faz |
|---|---|
| `"intercepto"` (padrão) | desloca o logit da PD até a PD média do dev igualar a taxa de maus do dev; não muda a ordem nem a inclinação |
| `"platt"` | regressão logística sobre o logit da PD: corrige nível e inclinação |
| `None` | sem calibração (a PD do modelo como está) |

O rating continua saindo da **PD original** com as faixas que você informou: cada rating mantém exatamente os mesmos contratos. A calibração muda só a linha "média do modelo" dos gráficos, o gap de calibração do backtest e a PD média das métricas. A régua (taxa observada no dev) e o semáforo não dependem dela. A tabela da régua mostra também os cortes convertidos para a escala calibrada, só para leitura.

**Modelo.** `MODELO_URI` aceita os dois formatos do MLflow: `"models:/<nome_registrado>/<versão>"` (modelo registrado) ou `"runs:/<run_id>/<caminho>"` (artefato de uma execução; o caminho costuma ser `model`).

- `TIPO_MODELO = "logistica"`: aceita scikit-learn, statsmodels e Spark ML. Se o modelo não puder ser lido, informe os coeficientes à mão em `COEFICIENTES_MANUAIS`.
- `TIPO_MODELO = "generico"`: aceita scikit-learn (inclusive pipelines), LightGBM e XGBoost; outros formatos passam pela interface genérica do MLflow (pyfunc), que precisa devolver probabilidade. As variáveis vêm da assinatura do modelo; se ela não existir, informe-as em `VARIAVEIS_MODELO`, na ordem do treino. A saída precisa ser uma PD entre 0 e 1 (as faixas são cortes de PD).

**Faixas.** Uma lista de `(rótulo, limite superior da PD)`, em ordem crescente de risco. O contrato entra no primeiro rating cujo limite é **maior** que a PD dele; o último rating usa `None` (sem limite). Exemplo com três ratings: `[("A", 0.01), ("B", 0.05), ("C", None)]` → PD 0,8% vai para A; 1,0% vai para B; 7% vai para C.

In [ ]:
# -----------------------------------------------------------------------------
# PARÂMETROS: edite aqui
# -----------------------------------------------------------------------------
CAMINHO_PARQUET = None                                  # ex.: "/Volumes/.../base_woe"
TABELA_ENTRADA = None                                   # ou uma tabela: "catalogo.esquema.base_woe"

MODELO_URI = "models:/<nome_do_modelo>/<versao>"        # ou "runs:/<run_id>/model"
TIPO_MODELO = "logistica"                               # "logistica" ou "generico"
COEFICIENTES_MANUAIS = None                             # só logística: {"intercepto": -3.0, "woe_x": 0.8, ...}
VARIAVEIS_MODELO = None                                 # só genérico, se o modelo não tiver assinatura: ["var1", "var2", ...]
CALIBRACAO_DEV = "intercepto"                           # "intercepto", "platt" ou None (sem calibração)

COLUNAS = {"safra": "data_ref", "alvo": "target", "amostra": "periodo"}
ROTULOS_PERIODO = {"dev": "dev", "oot": "oot"}          # valores da coluna de período
DATA_CORTE = "2026-09-01"                               # último mês com informação, com dia 01

FAIXAS = [                                              # (rótulo, limite superior da PD); último = None
    ("A", 0.005),
    ("B", 0.010),
    ("C", 0.020),
    ("D", 0.040),
    ("E", 0.080),
    ("F", None),
]

ESQUEMA_SAIDA = None                                    # ex.: "catalogo.esquema" (para GRAVAR)
ESQUEMA_TEMPORARIO = None                               # ex.: "catalogo.esquema": guarda a base escorada (evita reescorar a cada gráfico)
EXPERIMENTO_MLFLOW = None                               # ex.: "/Workspace/Users/<email>/<pasta>/avaliacao_modelo"
GRAVAR = False                                          # True = grava as tabelas de resultado no esquema

## 3. Configuração

Usa a mesma configuração do pipeline (tarefa PD), só com as colunas desta base. As tabelas gravadas ganham o prefixo `modelo_existente_pd_`.

In [ ]:
cfg = cria_configuracao(
    tarefa="pd", metrica="ks", data_corte=DATA_CORTE,
    esquema_saida=ESQUEMA_SAIDA if GRAVAR else None, experimento_mlflow=EXPERIMENTO_MLFLOW,
    ajustes={"prefixo_tabelas": "modelo_existente", "esquema_temporario": ESQUEMA_TEMPORARIO,
             "colunas": {**COLUNAS, "data_primeiro_default": None, "data_fim_observacao": None},
             "rotulos_amostra": {"des": ROTULOS_PERIODO["dev"], "oot": ROTULOS_PERIODO["oot"]},
             # sem divisão treino/teste aqui: a regra de recalibrar olha só as safras do OOT
             "backtest_regua": {"particoes_recalibracao": ["oot"]}})
print(f"id_execucao = {cfg['id_execucao']}")

## 4. Funções desta avaliação

In [ ]:
import time

import pandas as pd
from pyspark.sql import functions as F

N_FAIXAS_SCORE = 10_000      # histograma fino do score para KS, AUC e PSI (sem ordenar a base)


def prepara_base(sdf, cfg):
    """
    Base pronta para a avaliação: datas no dia 01, só dev e OOT, safras maduras
    e a coluna `particao` ("dev" ou "oot").
    """
    colunas, rotulos = cfg["colunas"], cfg["rotulos_amostra"]
    sdf = sdf.withColumn(colunas["safra"], F.trunc(F.col(colunas["safra"]), "month"))
    sdf = sdf.filter(F.col(colunas["amostra"]).isin(rotulos["des"], rotulos["oot"]))
    sdf = adiciona_colunas_derivadas(sdf, cfg)
    resumo = (sdf.groupBy(colunas["amostra"], "safra_madura").count().toPandas()
                 .sort_values([colunas["amostra"], "safra_madura"]))
    imaturas = int(resumo.loc[~resumo["safra_madura"], "count"].sum())
    if imaturas:
        print(f"[base] {imaturas:,} contratos em safras imaturas (menos de "
              f"{cfg['entrada']['janela_desempenho_meses']} meses até {cfg['data_corte']}) ficam de fora.")
    sdf = sdf.filter(~condicao_safra_imatura(cfg))
    amostra = F.col(colunas["amostra"])
    return sdf.withColumn("particao", F.when(amostra == rotulos["des"], "dev").otherwise("oot"))


def carrega_logistica(uri=None, coeficientes_manuais=None):
    """
    Lê intercepto e coeficientes de uma regressão logística salva no MLflow.

    Devolve {"intercepto", "coeficientes": {variável: coeficiente}, "tipo", "modelo"}.

    Conceito técnico:
        O MLflow guarda o modelo num "flavor" (formato): sklearn, statsmodels,
        spark... O flavor diz como reabrir o objeto original. Da logística só
        precisamos do intercepto e dos coeficientes: com eles a PD sai de uma
        conta simples, feita no próprio Spark.
    """
    if coeficientes_manuais:
        c = dict(coeficientes_manuais)
        intercepto = float(c.pop("intercepto"))
        return {"intercepto": intercepto, "coeficientes": {k: float(v) for k, v in c.items()},
                "tipo": "manual", "modelo": None}
    import mlflow
    informacao = mlflow.models.get_model_info(uri)
    flavors = set(informacao.flavors)
    nomes_assinatura = None
    if informacao.signature is not None and informacao.signature.inputs is not None:
        try:
            nomes_assinatura = informacao.signature.inputs.input_names()
        except Exception:
            nomes_assinatura = None

    if "sklearn" in flavors:
        import mlflow.sklearn
        modelo = mlflow.sklearn.load_model(uri)
        etapas = getattr(modelo, "steps", None)
        estimador = etapas[-1][1] if etapas else modelo
        if etapas and len(etapas) > 1:
            raise ValueError(
                f"O modelo é um Pipeline com transformações antes da logística ({[n for n, _ in etapas[:-1]]}). "
                "Os coeficientes valem para a saída dessas transformações, não para as WoE da base. "
                "Informe os coeficientes em COEFICIENTES_MANUAIS ou salve só a logística.")
        nomes = getattr(estimador, "feature_names_in_", None)
        nomes = list(nomes) if nomes is not None else nomes_assinatura
        coeficientes = np.ravel(estimador.coef_)
        intercepto = float(np.ravel(estimador.intercept_)[0])
        tipo = "sklearn"
    elif "statsmodels" in flavors:
        import mlflow.statsmodels
        modelo = mlflow.statsmodels.load_model(uri)
        parametros = pd.Series(modelo.params)
        chave = next((k for k in ("const", "Intercept", "intercept") if k in parametros.index), None)
        intercepto = float(parametros[chave]) if chave else 0.0
        parametros = parametros.drop(chave) if chave else parametros
        nomes, coeficientes, tipo = list(parametros.index), parametros.to_numpy(), "statsmodels"
    elif "spark" in flavors:
        import mlflow.spark
        modelo = mlflow.spark.load_model(uri)
        etapas = list(getattr(modelo, "stages", [modelo]))
        logistica = next(e for e in etapas if hasattr(e, "coefficients"))
        montador = next((e for e in etapas if type(e).__name__ == "VectorAssembler"), None)
        outras = [type(e).__name__ for e in etapas if e is not logistica and e is not montador]
        if outras:
            raise ValueError(f"O pipeline Spark tem outras etapas além da logística ({outras}); "
                             "informe os coeficientes em COEFICIENTES_MANUAIS.")
        nomes = montador.getInputCols() if montador is not None else nomes_assinatura
        coeficientes = logistica.coefficients.toArray()
        intercepto = float(logistica.intercept)
        tipo = "spark"
    else:
        raise ValueError(f"Formato do modelo não reconhecido ({sorted(flavors)}). "
                         "Informe os coeficientes em COEFICIENTES_MANUAIS.")
    if not nomes or len(nomes) != len(coeficientes):
        raise ValueError("Não foi possível ler os nomes das variáveis do modelo. "
                         "Informe os coeficientes em COEFICIENTES_MANUAIS.")
    return {"intercepto": intercepto, "coeficientes": dict(zip(nomes, map(float, coeficientes))),
            "tipo": tipo, "modelo": modelo}


def escora_logistica(sdf, logistica, coluna="pd_modelo"):
    """PD de cada contrato: 1 / (1 + e^−margem), com margem = intercepto + Σ coeficiente × variável."""
    faltam = [v for v in logistica["coeficientes"] if v not in sdf.columns]
    if faltam:
        raise ValueError(f"Variáveis do modelo que não estão na base: {faltam}")
    margem = F.lit(logistica["intercepto"])
    for variavel, coeficiente in logistica["coeficientes"].items():
        margem = margem + F.lit(coeficiente) * F.col(variavel).cast("double")
    return sdf.withColumn(coluna, 1 / (1 + F.exp(-margem)))


def confere_escoragem(base_escorada, logistica, n=2000, semente=42):
    """
    Compara a PD calculada no Spark com a previsão do próprio modelo numa amostra.
    A maior diferença deve ser da ordem de 1e-12 (só arredondamento).
    """
    if logistica["modelo"] is None or logistica["tipo"] == "spark":
        print("[conferência] coeficientes manuais ou modelo Spark: conferência pela própria fórmula, sem comparação externa.")
        return None
    variaveis = list(logistica["coeficientes"])
    amostra = base_escorada.select(*variaveis, "pd_modelo").limit(n).toPandas()
    X = amostra[variaveis].astype(float)
    if logistica["tipo"] == "sklearn":
        previsto = logistica["modelo"].predict_proba(X)[:, 1]
    else:
        import statsmodels.api as sm
        previsto = np.asarray(logistica["modelo"].predict(sm.add_constant(X, has_constant="add")))
    diferenca = float(np.nanmax(np.abs(previsto - amostra["pd_modelo"].to_numpy())))
    print(f"[conferência] maior diferença entre a PD do Spark e a do modelo em {len(amostra)} contratos: {diferenca:.2e}")
    return diferenca


def carrega_modelo_generico(uri, variaveis=None):
    """
    Carrega um modelo do MLflow pela biblioteca original (para chamar a probabilidade).

    Devolve {"modelo", "tipo", "variaveis"}.

    Conceito técnico:
        Pela interface genérica do MLflow (pyfunc), um classificador costuma
        devolver a CLASSE (0 ou 1), não a probabilidade. Por isso o modelo é
        aberto pelo "flavor" da biblioteca original (sklearn, lightgbm, xgboost),
        que dá acesso a predict_proba (ou à previsão de probabilidade do booster).
    """
    import mlflow
    informacao = mlflow.models.get_model_info(uri)
    flavors = set(informacao.flavors)
    nomes = None
    if informacao.signature is not None and informacao.signature.inputs is not None:
        try:
            nomes = informacao.signature.inputs.input_names()
        except Exception:
            nomes = None
    if "sklearn" in flavors:
        import mlflow.sklearn
        modelo, tipo = mlflow.sklearn.load_model(uri), "sklearn"
    elif "lightgbm" in flavors:
        import mlflow.lightgbm
        modelo, tipo = mlflow.lightgbm.load_model(uri), "lightgbm"
    elif "xgboost" in flavors:
        import mlflow.xgboost
        modelo, tipo = mlflow.xgboost.load_model(uri), "xgboost"
    else:
        modelo, tipo = mlflow.pyfunc.load_model(uri), "pyfunc"
        print("[modelo] formato sem acesso direto à probabilidade: usando a interface genérica do MLflow (pyfunc). "
              "Se ela devolver a classe 0/1, a escoragem para com erro.")
    if variaveis is None:
        variaveis = nomes
    if variaveis is None:
        for atributo in ("feature_names_in_", "feature_name_", "feature_names"):
            valor = getattr(modelo, atributo, None)
            if valor is not None and not callable(valor):
                variaveis = list(valor)
                break
        if variaveis is None and hasattr(modelo, "feature_name") and callable(modelo.feature_name):
            variaveis = list(modelo.feature_name())                   # Booster do LightGBM
    if not variaveis:
        raise ValueError("Não foi possível ler as variáveis do modelo. Informe-as em VARIAVEIS_MODELO.")
    return {"modelo": modelo, "tipo": tipo, "variaveis": list(variaveis)}


def preve_probabilidade(modelo, X):
    """PD (probabilidade da classe 1) de um lote pandas, conforme o tipo do objeto."""
    if hasattr(modelo, "predict_proba"):
        return np.asarray(modelo.predict_proba(X))[:, 1]
    tipo = type(modelo).__module__ + "." + type(modelo).__name__
    if tipo.startswith("lightgbm") and type(modelo).__name__ == "Booster":
        return np.asarray(modelo.predict(X), dtype=float)
    if tipo.startswith("xgboost") and type(modelo).__name__ == "Booster":
        import xgboost as xgb
        return np.asarray(modelo.predict(xgb.DMatrix(X)), dtype=float)
    previsto = np.asarray(modelo.predict(X), dtype=float).ravel()
    if set(np.unique(previsto)) <= {0.0, 1.0}:
        raise ValueError("O modelo devolveu só 0 e 1 (a classe, não a probabilidade). Salve o modelo com "
                         "acesso à probabilidade ou use um formato sklearn, lightgbm ou xgboost.")
    return previsto


def escora_generico(sdf, carregado, cfg, coluna="pd_modelo"):
    """
    PD de cada contrato com o próprio modelo, em lotes pandas nos workers (mapInPandas).
    Só seguem as colunas de controle (safra, alvo, período, partição e derivadas) e a PD.
    """
    from pyspark.sql.types import DecimalType, DoubleType, StructField, StructType
    variaveis = carregado["variaveis"]
    faltam = [v for v in variaveis if v not in sdf.columns]
    if faltam:
        raise ValueError(f"Variáveis do modelo que não estão na base: {faltam}")
    controle = set(colunas_de_controle(cfg)) | set(COLUNAS_DERIVADAS)
    manter = [c for c in sdf.columns if c in controle and c not in variaveis]
    tipos = dict((campo.name, campo.dataType) for campo in sdf.schema.fields)
    entrada = sdf.select(*manter, *[F.col(v).cast("double").alias(v) if isinstance(tipos[v], DecimalType) else F.col(v)
                                    for v in variaveis])
    esquema = StructType(list(sdf.select(*manter).schema.fields) + [StructField(coluna, DoubleType())])
    modelo = carregado["modelo"]

    def escora_lotes(lotes):
        for lote in lotes:
            saida = lote[manter].copy()
            saida[coluna] = preve_probabilidade(modelo, lote[variaveis]) if len(lote) else np.array([], dtype=float)
            yield saida

    return entrada.mapInPandas(escora_lotes, schema=esquema)


def confere_generico(base_escorada, cfg, coluna="pd_modelo"):
    """
    Sanidade da PD do modelo genérico: dentro de [0, 1], não constante e com a
    média perto da taxa observada. Para com erro se a PD estiver fora de [0, 1]
    ou for constante.
    """
    alvo = F.col(cfg["colunas"]["alvo"]).cast("double")
    r = (base_escorada.groupBy("particao")
         .agg(F.count(F.lit(1)).alias("contratos"), F.min(coluna).alias("pd_minima"), F.max(coluna).alias("pd_maxima"),
              F.avg(coluna).alias("pd_media"), F.avg(alvo).alias("taxa_observada"),
              F.stddev(coluna).alias("desvio_pd")).toPandas())
    exibe(r, "Conferência da PD do modelo (sanidade)")
    if (r["pd_minima"] < 0).any() or (r["pd_maxima"] > 1).any():
        raise ValueError("A saída do modelo não está entre 0 e 1: não é uma PD (veja a tabela de conferência).")
    if (r["desvio_pd"].fillna(0) == 0).any():
        raise ValueError("A PD do modelo é constante em algum período: confira as variáveis de entrada.")
    razao = (r["pd_media"] / r["taxa_observada"]).replace([np.inf, -np.inf], np.nan)
    if ((razao < 0.5) | (razao > 2)).any():
        print("[conferência] ATENÇÃO: a PD média está a mais de 2 vezes da taxa observada em algum período "
              "(modelo descalibrado ou variáveis em formato diferente do treino).")
    return r


def _logit_dev(p):
    """logit(p) = ln(p / (1 − p)), com p limitado a [1e-6, 1 − 1e-6]."""
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def ajusta_calibracao_dev(base_escorada, cfg, metodo, max_linhas=1_000_000):
    """
    Ajusta a calibração da PD no dev. Devolve um dicionário com a regra (ou None, se metodo for None).

    Conceito técnico:
        Uma amostra do dev (até max_linhas contratos; só a PD e o alvo) vai para o
        pandas. Intercepto: procura δ tal que a média de 1 / (1 + e^−(logit(PD) + δ))
        seja a taxa de maus do dev (método de Brent). Platt: regressão logística
        sem regularização do alvo sobre logit(PD), que dá a inclinação a e o intercepto b.
    """
    if metodo is None:
        return None
    if metodo not in ("intercepto", "platt"):
        raise ValueError('CALIBRACAO_DEV deve ser "intercepto", "platt" ou None.')
    dev = base_escorada.filter(F.col("particao") == "dev").select(
        F.col("pd_modelo").cast("double").alias("p"), F.col(cfg["colunas"]["alvo"]).cast("double").alias("y"))
    total = dev.count()
    if total > max_linhas:
        dev = dev.sample(fraction=max_linhas / total, seed=cfg["semente"])
    amostra = dev.toPandas().dropna()
    z, y = _logit_dev(amostra["p"].to_numpy()), amostra["y"].to_numpy()
    if metodo == "intercepto":
        from scipy.optimize import brentq
        delta = float(brentq(lambda d: float(np.mean(1 / (1 + np.exp(-(z + d))))) - float(y.mean()), -20, 20))
        regra = {"metodo": "intercepto", "a": 1.0, "b": delta}
    else:
        from sklearn.linear_model import LogisticRegression
        ajuste = LogisticRegression(C=1e12, max_iter=1000).fit(z.reshape(-1, 1), y.astype(int))
        regra = {"metodo": "platt", "a": float(ajuste.coef_[0, 0]), "b": float(ajuste.intercept_[0])}
    return dict(regra, n_amostra=int(len(amostra)))


def calibra_pd(p, calibracao):
    """PD calibrada (numpy): 1 / (1 + e^−(a · logit(p) + b)). Sem calibração, devolve p."""
    p = np.asarray(p, dtype=float)
    if calibracao is None:
        return p
    return 1 / (1 + np.exp(-(calibracao["a"] * _logit_dev(p) + calibracao["b"])))


def aplica_calibracao_dev(sdf, calibracao, entrada="pd_modelo", saida="score_calibrado"):
    """Mesma regra de calibra_pd numa coluna Spark (todas as safras, dev e OOT)."""
    if calibracao is None:
        return sdf.withColumn(saida, F.col(entrada))
    p = F.least(F.greatest(F.col(entrada).cast("double"), F.lit(1e-6)), F.lit(1 - 1e-6))
    z = F.lit(calibracao["a"]) * F.log(p / (1 - p)) + F.lit(calibracao["b"])
    return sdf.withColumn(saida, 1 / (1 + F.exp(-z)))


def tabela_calibracao_dev(base_calibrada, cfg):
    """Por período: contratos, taxa de maus, PD média do modelo antes e depois da calibração."""
    alvo = F.col(cfg["colunas"]["alvo"]).cast("double")
    return (base_calibrada.groupBy("particao")
            .agg(F.count(F.lit(1)).alias("contratos"), F.avg(alvo).alias("taxa_maus"),
                 F.avg("pd_modelo").alias("pd_media_antes"), F.avg("score_calibrado").alias("pd_media_depois"))
            .toPandas().sort_values("particao").reset_index(drop=True))


def regua_das_faixas(faixas):
    """Esqueleto da régua (rótulos e cortes) a partir da lista (rótulo, limite superior)."""
    limites = [l for _, l in faixas[:-1]]
    if faixas[-1][1] is not None:
        raise ValueError("O último rating deve ter limite None (sem limite superior).")
    if any(l is None for l in limites) or limites != sorted(limites):
        raise ValueError("Os limites das faixas devem ser crescentes, e só o último pode ser None.")
    regua = pd.DataFrame({"rating": [r for r, _ in faixas],
                          "corte_superior": [float(l) for l in limites] + [None]})
    regua["corte_inferior"] = [None] + regua["corte_superior"].tolist()[:-1]
    regua["valor_regua"] = 0.0                     # provisório: preenchido com a taxa do dev
    return regua


def completa_regua(regua, agregado, cfg):
    """Valor de cada rating = taxa de maus observada no dev, com intervalo e participação."""
    dev = agregado[agregado["particao"] == "dev"].groupby("rating")[["n", "soma", "soma2", "soma_score"]].sum()
    oot = agregado[agregado["particao"] == "oot"].groupby("rating")[["n", "soma"]].sum()
    regua = regua.copy()
    linhas = []
    for rating in regua["rating"]:
        e = dev.loc[rating].to_dict() if rating in dev.index else {"n": 0, "soma": 0, "soma2": 0, "soma_score": 0}
        ic_inf, ic_sup = intervalo(e, "pd")
        o = oot.loc[rating] if rating in oot.index else {"n": 0, "soma": 0}
        linhas.append({"n_des": int(e["n"]), "valor_regua": e["soma"] / e["n"] if e["n"] else np.nan,
                       "ic_inf": ic_inf, "ic_sup": ic_sup,
                       "score_medio_des": e["soma_score"] / e["n"] if e["n"] else np.nan,
                       "n_oot": int(o["n"]), "observado_oot": o["soma"] / o["n"] if o["n"] else np.nan})
    regua = pd.concat([regua.drop(columns=["valor_regua"]), pd.DataFrame(linhas)], axis=1)
    regua["participacao_des"] = regua["n_des"] / regua["n_des"].sum()
    taxas = regua["valor_regua"].to_numpy()
    if np.any(np.diff(taxas[~np.isnan(taxas)]) < 0):
        print("[faixas] ATENÇÃO: a taxa de maus do dev não cresce de um rating para o seguinte "
              "em algum ponto (inversão). Veja a coluna valor_regua.")
    vazios = regua.loc[regua["n_des"] == 0, "rating"].tolist()
    if vazios:
        print(f"[faixas] ATENÇÃO: ratings sem contratos no dev: {vazios}")
    return regua


def metricas_por_periodo(base_escorada, cfg, coluna="pd_modelo", coluna_calibrada="score_calibrado"):
    """
    KS, AUC, Gini, taxa de maus, PD média (original e calibrada) e PSI do score, por período (dev e OOT).
    KS, AUC e PSI usam a PD original (a calibração não muda a ordem dos contratos).

    Conceito técnico:
        Tudo sai de um histograma do score em 10.000 faixas por período (uma
        agregação Spark). AUC = probabilidade de um mau sorteado ter PD maior
        que um bom sorteado (empates contam meio). Gini = 2 × AUC − 1.
        PSI: o score do OOT contra os decis do score do dev.
    """
    alvo = F.col(cfg["colunas"]["alvo"]).cast("double")
    faixa = F.least(F.floor(F.col(coluna) * N_FAIXAS_SCORE).cast("int"), F.lit(N_FAIXAS_SCORE - 1))
    h = (base_escorada.groupBy("particao", faixa.alias("faixa"))
                      .agg(F.count(F.lit(1)).alias("n"), F.sum(alvo).alias("maus"),
                           F.sum(F.col(coluna)).alias("soma_pd"),
                           F.sum(F.col(coluna_calibrada)).alias("soma_pd_calibrada")).toPandas())
    dev = h[h["particao"] == "dev"].sort_values("faixa")
    acumulado = dev["n"].cumsum() / dev["n"].sum()
    decil_da_faixa = dict(zip(dev["faixa"], np.minimum((acumulado.to_numpy() * 10 - 1e-9).astype(int), 9)))
    linhas = []
    for particao in ("dev", "oot"):
        g = h[h["particao"] == particao].sort_values("faixa")
        if g.empty:
            continue
        maus, bons = g["maus"].to_numpy(), (g["n"] - g["maus"]).to_numpy()
        total_maus, total_bons = maus.sum(), bons.sum()
        bons_abaixo = np.concatenate([[0], np.cumsum(bons)[:-1]])
        auc = float((maus * (bons_abaixo + 0.5 * bons)).sum() / (total_maus * total_bons)) \
            if total_maus and total_bons else np.nan
        linhas.append({"periodo": particao, "contratos": int(g["n"].sum()), "maus": int(total_maus),
                       "taxa_maus": total_maus / g["n"].sum(), "pd_media": g["soma_pd"].sum() / g["n"].sum(),
                       "pd_media_calibrada": g["soma_pd_calibrada"].sum() / g["n"].sum(),
                       "ks": _ks_de_histograma(g.rename(columns={"maus": "eventos"})),
                       "auc": auc, "gini": 2 * auc - 1})
    tabela = pd.DataFrame(linhas)
    oot = h[h["particao"] == "oot"]
    if len(oot) and len(dev):
        e = dev.groupby(dev["faixa"].map(decil_da_faixa))["n"].sum() / dev["n"].sum()
        decis_oot = oot["faixa"].map(lambda f: decil_da_faixa.get(f, 0 if f < dev["faixa"].min() else 9))
        a = oot.groupby(decis_oot)["n"].sum() / oot["n"].sum()
        a = a.reindex(e.index, fill_value=0.0)
        psi = float(((a - e) * np.log((a + 1e-6) / (e + 1e-6))).sum())
        tabela["psi_score_vs_dev"] = tabela["periodo"].map({"dev": 0.0, "oot": psi})
    return tabela

## 5. Base de entrada

Acrescente aqui os tratamentos da sua base, se houver (filtros, minúsculas no período).

In [ ]:
if CAMINHO_PARQUET:
    sdf_entrada = spark.read.parquet(CAMINHO_PARQUET)
elif TABELA_ENTRADA:
    sdf_entrada = spark.table(TABELA_ENTRADA)
else:
    raise ValueError("Preencha CAMINHO_PARQUET ou TABELA_ENTRADA.")

# Tratamentos da sua base (exemplo):
# sdf_entrada = sdf_entrada.withColumn("periodo", F.lower(F.trim(F.col("periodo"))))

base = prepara_base(sdf_entrada, cfg)

## 6. Modelo e escoragem

In [ ]:
if TIPO_MODELO == "logistica":
    logistica = carrega_logistica(MODELO_URI, COEFICIENTES_MANUAIS)
    exibe(pd.DataFrame({"variavel": ["intercepto", *logistica["coeficientes"]],
                        "coeficiente": [logistica["intercepto"], *logistica["coeficientes"].values()]}),
          f"Regressão logística ({logistica['tipo']}): {len(logistica['coeficientes'])} variáveis")
    base_escorada = escora_logistica(base, logistica)
    confere_escoragem(base_escorada, logistica)
    descricao_modelo = {"tipo_modelo": f"logistica ({logistica['tipo']})", "n_variaveis": len(logistica["coeficientes"])}
elif TIPO_MODELO == "generico":
    carregado = carrega_modelo_generico(MODELO_URI, VARIAVEIS_MODELO)
    print(f"[modelo] {carregado['tipo']}: {len(carregado['variaveis'])} variáveis")
    base_escorada = escora_generico(base, carregado, cfg)
    # Guarda a base escorada (se houver esquema temporário): sem isso, cada gráfico reescora a base.
    base_escorada = guarda_resultado(spark, base_escorada, "base_escorada", cfg)
    confere_generico(base_escorada, cfg)
    descricao_modelo = {"tipo_modelo": carregado["tipo"], "n_variaveis": len(carregado["variaveis"])}
else:
    raise ValueError('TIPO_MODELO deve ser "logistica" ou "generico".')

## 6b. Calibração da PD no dev

A regra é ajustada no dev e aplicada a todas as safras. A tabela mostra, por período, a taxa de maus e a PD média do modelo antes e depois: no dev, com o intercepto, a PD média depois fica igual à taxa; no OOT, a distância entre as duas mostra o deslocamento do nível do modelo fora do dev.

In [ ]:
calibracao_dev = ajusta_calibracao_dev(base_escorada, cfg, CALIBRACAO_DEV)
base_escorada = aplica_calibracao_dev(base_escorada, calibracao_dev)
if calibracao_dev is None:
    print("[calibração] desligada: a média do modelo nos gráficos é a PD original.")
else:
    contratos = f"{calibracao_dev['n_amostra']:,}".replace(",", ".")
    print(f"[calibração] {calibracao_dev['metodo']} no dev: a = {calibracao_dev['a']:.4f}; "
          f"b = {calibracao_dev['b']:.4f} ({contratos} contratos)")
tabela_calibracao = tabela_calibracao_dev(base_escorada, cfg)
exibe(tabela_calibracao, "Calibração no dev: taxa de maus × PD média do modelo antes e depois")

## 7. Métricas por período

In [ ]:
metricas = metricas_por_periodo(base_escorada, cfg)
exibe(metricas, "KS, AUC, Gini, taxa de maus, PD média e PSI do score por período")

## 8. Faixas de rating e backtest

O rating de cada contrato sai das faixas informadas. O valor de cada rating (a "régua") é a taxa de maus observada no dev. O monitoramento é o mesmo do pipeline (`monitora_regua`, notebook 05): semáforo por rating e safra (verde dentro do IC 95%, amarelo fora do 95% e dentro do 99%, vermelho fora do 99%) e recalibração avaliada **só nas safras do OOT** (aqui não há teste): 3 vermelhos seguidos, ou 4 seguidas fora do verde.

In [ ]:
esqueleto = regua_das_faixas(FAIXAS)
# Rating pela PD ORIGINAL com as faixas informadas: cada rating mantém os mesmos contratos.
base_com_rating = aplica_regua(base_escorada, esqueleto, coluna_score="pd_modelo")
agregado = agrega_backtest(base_com_rating, cfg)
regua = completa_regua(esqueleto, agregado, cfg)
# Cortes na escala da PD calibrada, só para leitura (a regra é monotônica: mesmos contratos em cada rating).
regua["corte_superior_calibrado"] = [None if c is None or pd.isna(c) else float(calibra_pd(c, calibracao_dev))
                                     for c in regua["corte_superior"]]
valor_do_rating = dict(zip(regua["rating"], regua["valor_regua"]))
agregado["soma_regua"] = agregado["n"] * agregado["rating"].map(valor_do_rating)

monitoramento = monitora_regua(base_com_rating, regua, cfg, agregado=agregado)
backtest, estabilidade = monitoramento["backtest"], monitoramento["estabilidade"]

exibe(regua[["rating", "corte_inferior", "corte_superior", "corte_superior_calibrado", "n_des", "participacao_des",
             "valor_regua", "ic_inf", "ic_sup", "score_medio_des", "n_oot", "observado_oot"]],
      "Faixas de rating (régua do dev; score_medio_des = PD calibrada média)")
exibe(backtest["rating_periodo"], "Backtest: rating por período")
exibe(backtest["total_periodo"], "Backtest: carteira por período")
exibe(backtest["recalibracao"], "Recalibração (OOT)")
marcados = backtest["recalibracao"][backtest["recalibracao"]["recalibrar"]]
regra = cfg["backtest_regua"]
print("[monitoramento] " + ("recalibrar: " + ", ".join(f"{r.rating} (desde {_rotulo_safra(r.desde)})" for r in marcados.itertuples())
                            if len(marcados) else
                            f"nenhum rating a recalibrar (regra: {regra['n_vermelhos_seguidos']} vermelhos seguidos ou "
                            f"{regra['n_fora_do_verde_seguidos']} fora do verde seguidos, no OOT)"))

## 9. Gráficos do monitoramento

1. **Ratings com semáforo:** um gráfico por rating, ponto de cada safra colorido pelo semáforo.
2. **Mapa de status:** rating × safra, com contratos e maus em cada célula.
3. **Volumetria:** contratos e maus empilhados por rating e safra.
4. **Painel de estabilidade:** taxa por rating e safra, participação com PSI, KS por safra com IC e a média de cada período, ordenação.
5. **Ponte da carteira** e **ordem por safra**.

In [ ]:
exibe(estabilidade, "Estabilidade por safra")
display(monitoramento["figura_ratings"])

In [ ]:
display(monitoramento["figura_mapa_status"])
display(monitoramento["figura_volumetria"])

In [ ]:
display(monitoramento["figura_estabilidade"])

In [ ]:
display(monitoramento["figura_ponte"])
display(monitoramento["figura_ordem"])

## 10. Registro no MLflow e gravação

Com `EXPERIMENTO_MLFLOW`, cria uma execução com os coeficientes, as tabelas (CSV) e os gráficos. Com `GRAVAR = True`, grava as tabelas no esquema, com o prefixo `modelo_existente_pd_`.

In [ ]:
tabelas = {"metricas_periodo": metricas, "regua": regua, "estabilidade_safra": estabilidade,
           "calibracao_dev": tabela_calibracao,
           **{f"backtest_{nome}": tabela for nome, tabela in backtest.items()}}
if TIPO_MODELO == "logistica":
    tabelas["coeficientes"] = pd.DataFrame({"variavel": list(logistica["coeficientes"]),
                                            "coeficiente": list(logistica["coeficientes"].values())})

if EXPERIMENTO_MLFLOW:
    import mlflow
    mlflow.set_experiment(EXPERIMENTO_MLFLOW)
    with mlflow.start_run(run_name=f"modelo_existente_{cfg['id_execucao']}"):
        mlflow.set_tags({"id_execucao": cfg["id_execucao"], "modelo_uri": str(MODELO_URI),
                         "tipo_modelo": descricao_modelo["tipo_modelo"]})
        mlflow.log_params({"n_variaveis": descricao_modelo["n_variaveis"], "faixas": json.dumps(FAIXAS),
                           "calibracao_dev": json.dumps(calibracao_dev),
                           **({"intercepto": logistica["intercepto"]} if TIPO_MODELO == "logistica" else {})})
        mlflow.log_metrics({f"{c}_{l.periodo}": float(getattr(l, c)) for l in metricas.itertuples()
                            for c in ("ks", "auc", "gini") if pd.notna(getattr(l, c))})
        salva_pacote_ponto("avaliacao", tabelas)
        for nome in ("figura_estabilidade", "figura_ratings", "figura_mapa_status", "figura_volumetria",
                     "figura_ponte", "figura_ordem"):
            registra_figura_mlflow(monitoramento[nome], f"avaliacao/{nome.replace('figura_', '')}.png")
    print("[mlflow] execução registrada.")

if GRAVAR:
    for nome, tabela in tabelas.items():
        grava_tabela_delta(spark, tabela, nome, cfg)

## 11. Limpeza

Apaga as tabelas temporárias desta avaliação (a base escorada, se `ESQUEMA_TEMPORARIO` foi informado).

In [ ]:
apaga_tabelas_temporarias(spark, cfg)